In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import io

def load_and_visualize_parquet(train_path, test_path, num_samples=5):
    # 读取 parquet 文件
    train_df = pd.read_parquet(train_path)
    test_df = pd.read_parquet(test_path)

    # 检查数据
    print("训练集信息:")
    print(train_df.info())
    print("测试集信息:")
    print(test_df.info())

    # 解析图像数据
    def parse_image(image_dict):
        """
        从字典中提取图像数据
        """
        # 假设字典中包含 "bytes" 键，表示图像的字节数据
        image_bytes = image_dict["bytes"]
        # 使用 PIL 加载字节数据为图像
        image = Image.open(io.BytesIO(image_bytes))
        return np.array(image)

    # 提取图像和标签
    train_images = [parse_image(img) for img in train_df['image']]
    train_labels = train_df['label'].values
    test_images = [parse_image(img) for img in test_df['image']]
    test_labels = test_df['label'].values

    # 可视化部分训练集样本
    plt.figure(figsize=(10, 2))
    for i in range(num_samples):
        plt.subplot(1, num_samples, i + 1)
        plt.imshow(train_images[i].astype(np.uint8))
        plt.title(f"Label: {train_labels[i]}")
        plt.axis('off')
    plt.show()

    # 可视化部分测试集样本
    plt.figure(figsize=(10, 2))
    for i in range(num_samples):
        plt.subplot(1, num_samples, i + 1)
        plt.imshow(test_images[i].astype(np.uint8))
        plt.title(f"Label: {test_labels[i]}")
        plt.axis('off')
    plt.show()


# 示例文件路径
train_path = "./mnistm_train.parquet"
test_path = "./mnistm_test.parquet"

# 加载和可视化数据
load_and_visualize_parquet(train_path, test_path)

# 处理HR数据集
HR的原始数据在Data里

## 处理train

In [ ]:
# 读取./Data中的数据
import os
import pandas as pd
train_path="./Data/train"
train_list = []
for file in os.listdir(train_path):
    train_list.append(file)

# 将trian_list中每一个文件读取为dataframe
train_df = pd.DataFrame()
for file in train_list:
    df = pd.read_csv(os.path.join(train_path, file))
    if train_df.empty:
        train_df = df
    else:
        # 按照id与train.df进行join
        train_df = pd.merge(train_df, df, on='id')

# 移动income列为最后一列目标列
train_df = train_df[[col for col in train_df if col != 'Income'] + ['Income']]

#保存为csv文件
train_df.to_csv('./data/HR/train.csv', index=False)

## 处理test

In [ ]:
# 读取./Data中的数据
import os
import pandas as pd
test_path="./Data/test"
test_list = []
for file in os.listdir(test_path):
    test_list.append(file)

# 将test_list中每一个文件读取为dataframe
test_df = pd.DataFrame()
for file in test_list:
    df = pd.read_csv(os.path.join(test_path, file))
    if test_df.empty:
        test_df = df
    else:
        # 按照id与test.df进行join
        test_df = pd.merge(test_df, df, on='id')

# 移动income列为最后一列目标列
test_df = test_df[[col for col in test_df if col != 'Income'] + ['Income']]

#保存为csv文件
test_df.to_csv('./data/HR/test.csv', index=False)

## 处理为不同工作类别的数据

In [ ]:
# 读取./Data中的数据
import os
import pandas as pd
full_train_path = "./data/HR/full_train.csv"
train_df = pd.read_csv(full_train_path)

# 删除workclass=6的行
train_df = train_df[train_df['Workclass'] != 6]

# 统计workclass列的频次
workclass_freq = train_df['Workclass'].value_counts()
print(workclass_freq)

# 验证集为workclass=4的数据
valid_df = train_df[train_df['Workclass'] == 4]
train_df = train_df[train_df['Workclass'] != 4]

#删除workclass列，id列
train_df = train_df.drop(columns=['Workclass', 'id'])
valid_df = valid_df.drop(columns=['Workclass', 'id'])
# 保存为csv文件
train_df.to_csv('./data/HR/train.csv', index=False)
valid_df.to_csv('./data/HR/valid.csv', index=False)

In [ ]:
# 读取./Data中的数据
import os
import pandas as pd
full_test_path = "./data/HR/full_test.csv"
test_df = pd.read_csv(full_test_path)

# 删除workclass=6的行
test_df = test_df[test_df['Workclass'] != 6]

#测试集为workclass=4的数据
test_df = test_df[test_df['Workclass'] == 4]
test_df = test_df.drop(columns=['Workclass', 'id'])
# 保存为csv文件
test_df.to_csv('./data/HR/test.csv', index=False)

In [ ]:
# 读取./Data中的数据
import os
import pandas as pd
full_test_path = "./data/HR/test.csv"
test_df = pd.read_csv(full_test_path)

test_df.head(5)

# 处理House数据集

In [ ]:
# 读取./Data中的数据
import os
import pandas as pd
train_path="/data/dengqiyan/EVA/merged_files.csv"
train_list = []

df_train = pd.read_csv(train_path)
df_train.head(5)

In [ ]:
# 提取所有具有NAN值的行
nan_rows = df_train[df_train.isnull().any(1)]
nan_rows

In [ ]:
# 提取City列中的所有唯一值
cities = df_train['City'].unique()
print(cities)

In [ ]:
from sklearn.model_selection import train_test_split

# 提取City中为Mumbai的行
mumbai_rows = df_train[df_train['City'] == 'Mumbai']

# 移动income列为最后一列目标列
mumbai_rows = mumbai_rows[[col for col in mumbai_rows if col != 'Price'] + ['Price']]

mumbai_rows.head(5)

In [ ]:

# 随机划分为50%的测试集和50%的验证集
mumbai_test, mumbai_validation = train_test_split(mumbai_rows, test_size=0.5, random_state=42)

# 保存为csv
mumbai_validation.to_csv("/data/dengqiyan/EVA/data/House/val.csv", index=False)
mumbai_test.to_csv("/data/dengqiyan/EVA/data/House/test.csv", index=False)

In [ ]:
from sklearn.model_selection import train_test_split

# 提取City中为Mumbai的行
other_rows = df_train[df_train['City'] != 'Mumbai']

# 移动income列为最后一列目标列
other_rows = other_rows[[col for col in other_rows if col != 'Price'] + ['Price']]

other_cities = other_rows['City'].unique()
print(other_cities)

In [ ]:
other_rows.to_csv("/data/dengqiyan/EVA/data/House/train.csv", index=False)

In [ ]:
import pandas as pd

def convert_to_one_hot(input_csv_path, output_csv_path):
    """
    将 CSV 文件中的所有非数值列（object/category）转换为 One-Hot 编码并保存为新的 CSV 文件。
    
    Args:
        input_csv_path (str): 输入 CSV 文件路径。
        output_csv_path (str): 输出 CSV 文件路径。
    """
    # 读取 CSV 文件，并指定不将第一列当做索引
    data = pd.read_csv(input_csv_path, index_col=False)
    
    # 如果出现 “Unnamed: 0” 这种多余索引列，可在此处删除
    # 你也可以判断一下如果它在 columns 里再删除，以避免报错
    if "Unnamed: 0" in data.columns:
        data.drop(columns=["Unnamed: 0"], inplace=True)
    
    # 有些情况下，CSV 里会有重复列名，这里先去重
    # （保留第一次出现的列，丢弃之后重复的同名列）
    if data.columns.duplicated().any():
        print("检测到重复列名，自动去重。")
        data = data.loc[:, ~data.columns.duplicated()]
    
    # 确定需要编码的非数值列：object 和 category 类型
    non_numeric_columns = data.select_dtypes(include=['object', 'category']).columns
    
    if len(non_numeric_columns) > 0:
        print(f"以下列是非数值列，将进行 One-Hot 编码：{list(non_numeric_columns)}")
        # 对非数值列进行 One-Hot 编码
        # drop_first=True 可以减少 dummy 变量陷阱，但会少一列参考
        # 可根据自己需求选择
        data = pd.get_dummies(data, 
                              columns=non_numeric_columns, 
                              drop_first=False)
    else:
        print("没有非数值列，不需要 One-Hot 编码。")

    # 保存为新的 CSV 文件
    data.to_csv(output_csv_path, index=False)
    print(f"处理完成，文件已保存为：{output_csv_path}")


input_csv = "/data/dengqiyan/EVA/data/House/val0.csv"   # 输入文件路径
output_csv = "/data/dengqiyan/EVA/data/House/val.csv" # 输出文件路径

convert_to_one_hot(input_csv, output_csv)

In [ ]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

def convert_with_label_encoding(input_csv_path, output_csv_path):
    # 读取 CSV
    data = pd.read_csv(input_csv_path, index_col=False)

    # 如果有多余的 "Unnamed: 0" 索引列则删除
    if "Unnamed: 0" in data.columns:
        data.drop(columns=["Unnamed: 0"], inplace=True)

    # 找出所有 object 或 category 类型的列
    non_numeric_columns = data.select_dtypes(include=["object", "category"]).columns.tolist()

    # 先对 'Location' 做 Label Encoding（如果有）
    if "Location" in non_numeric_columns:
        lbl = LabelEncoder()
        data["Location"] = lbl.fit_transform(data["Location"])
        # 做完后，从要做 get_dummies 的列表里移除
        non_numeric_columns.remove("Location")

    # 对剩余的非数值列逐一做 One-Hot 编码
    # 并把新列插入到原列的位置，实现“在原列替换”效果
    for col in non_numeric_columns:
        # dummies: 为当前列生成哑变量
        dummies = pd.get_dummies(data[col], prefix=col, drop_first=False)

        # 找出当前列在 DataFrame 中的列索引位置
        col_idx = data.columns.get_loc(col)

        # 删掉原列
        data.drop(columns=[col], inplace=True)

        # 将哑变量插入到原列的位置
        # 把 data 拆分成“左侧列 + 右侧列”，再把新的 dummies 拼接中间
        left_part = data.iloc[:, :col_idx]
        right_part = data.iloc[:, col_idx:]
        data = pd.concat([left_part, dummies, right_part], axis=1)

    # 导出到 CSV
    data.to_csv(output_csv_path, index=False)
    print(f"文件保存：{output_csv_path}")

input_csv = "/data/dengqiyan/EVA/data/House/test0.csv"   # 输入文件路径
output_csv = "/data/dengqiyan/EVA/data/House/test.csv" # 输出文件路径
convert_with_label_encoding(input_csv, output_csv)

## 删除因为转one-hot导致的多余列

In [ ]:
import pandas as pd

df_file = pd.read_csv("/data/dengqiyan/EVA/data/House/valid.csv")
# 删除列City_Mumbai
df_file = df_file.drop(columns=['City_Mumbai'])
df_file.head(2)
df_file.to_csv("/data/dengqiyan/EVA/data/House/valid.csv", index=False)

In [ ]:
import pandas as pd

df_file2 = pd.read_csv("/data/dengqiyan/EVA/data/House/train.csv")
# 删除列City_Bangalore	City_Chennai	City_Delhi	City_Hyderabad	City_Kolkata
df_file2 = df_file2.drop(columns=['City_Bangalore', 'City_Chennai', 'City_Delhi', 'City_Hyderabad', 'City_Kolkata'])
df_file2.head(2)
df_file2.to_csv("/data/dengqiyan/EVA/data/House/train.csv", index=False)

In [ ]:
import pandas as pd

df_file = pd.read_csv("/data/dengqiyan/EVA/data/House/test.csv")
# 删除列City_Mumbai
df_file = df_file.drop(columns=['City_Mumbai'])
df_file.to_csv("/data/dengqiyan/EVA/data/House/test.csv", index=False)
df_file.head(2)

In [ ]:
import pandas as pd
import numpy as np
# 读取/data/dengqiyan/EVA/data/IMDB/IMDBLargeC5-train-X.npy文件
train = np.load("/data/dengqiyan/EVA/data/IMDB/IMDBLargeC5-train-X.npy")

train_Xdf = pd.DataFrame(train)
print(train_Xdf.head(5))

# 读取/data/dengqiyan/EVA/data/IMDB/IMDBLargeC5-train-y.npy文件
train_y = np.load("/data/dengqiyan/EVA/data/IMDB/IMDBLargeC5-train-y.npy")
train_ydf = pd.DataFrame(train_y)
print(train_ydf.head(5))

# 处理ImageNet数据集

In [ ]:
# 读入.pkl文件
import pickle
data, labels = pickle.load(open("/data/dengqiyan/EVA/data/TinyImageNet/val.pkl", "rb"))
print(len(data), len(labels))
labels = labels.long()  # 转换为 long 类型，即整数类型
# 按照labels的类别均分为valid.pkl和test.pkl
from sklearn.model_selection import train_test_split
data_valid, data_test, labels_valid, labels_test = train_test_split(data, labels, test_size=0.5, random_state=42)
print(len(data_valid), len(labels_valid))
print(len(data_test), len(labels_test))
# 保存为valid.pkl和test.pkl
valid_path = "/data/dengqiyan/EVA/data/TinyImageNet/valid.pkl"
test_path = "/data/dengqiyan/EVA/data/TinyImageNet/test.pkl"
pickle.dump((data_valid, labels_valid), open(valid_path, "wb"))
pickle.dump((data_test, labels_test), open(test_path, "wb"))

# 输出labels的类别个数
import numpy as np
import pandas as pd
# 读取/data/dengqiyan/EVA/data/TinyImageNet/val.pkl文件
data, labels = pickle.load(open("/data/dengqiyan/EVA/data/TinyImageNet/valid.pkl", "rb"))
# 输出labels的类别个数
print(len(np.unique(labels)))
# 输出labels
print(labels)

In [ ]:
# 读入.pkl文件
import pickle
data, labels = pickle.load(open("/data/dengqiyan/EVA/data/TinyImageNet/train0.pkl", "rb"))
print(len(data), len(labels))
labels = labels.long()  # 转换为 long 类型，即整数类型

train_path = "/data/dengqiyan/EVA/data/TinyImageNet/train.pkl"
pickle.dump((data, labels), open(train_path, "wb"))


# 输出labels的类别个数
import numpy as np
import pandas as pd
# 读取/data/dengqiyan/EVA/data/TinyImageNet/val.pkl文件
data, labels = pickle.load(open("/data/dengqiyan/EVA/data/TinyImageNet/train.pkl", "rb"))
# 输出labels的类别个数
print(len(np.unique(labels)))
# 输出labels
print(labels)

In [ ]:
import pandas as pd
import joblib
import numpy as np

samples_path = "/data/dengqiyan/wyh-dataset/tabular-dataset/covertype/samples_py3"
targets_path = "/data/dengqiyan/wyh-dataset/tabular-dataset/covertype/targets_py3"

# 加载数据
X = joblib.load(samples_path)
Y = joblib.load(targets_path)

# 如果需要像 DataFrame 那样查看数据，可以转换为 DataFrame
X_df = pd.DataFrame(X)
Y_df = pd.DataFrame(Y)
Y_df.head(5)

def get_integer_columns(X, tol=1e-5):
    """
    判断 X 中哪些列为整数列。

    参数:
        X (np.ndarray): 形状为 (n_samples, n_features) 的数组
        tol (float): 容差，若每个数值与其四舍五入后的差小于 tol，则认为该数值为整数

    返回:
        int_cols (list): 整数列的列索引列表
    """
    int_cols = []
    for i in range(X.shape[1]):
        col = X[:, i]
        # 判断该列所有数值是否接近整数
        if np.all(np.abs(col - np.round(col)) < tol):
            int_cols.append(i)
    return int_cols

integer_indices = get_integer_columns(X)

print("整数列的索引为:", integer_indices)

# 如果需要查看这些列的内容，可以转换为 DataFrame 查看前几行：
X_int = pd.DataFrame(X[:, integer_indices])
X_int.head()

In [ ]:
import pandas as pd
import joblib
import numpy as np

X_path = "/data/dengqiyan/EVA/data/IMDB/IMDBLargeC5-train-X.npy"
Y_path = "/data/dengqiyan/EVA/data/IMDB/IMDBLargeC5-test-y.npy"

# 加载数据
X = np.load(X_path)
Y = np.load(Y_path)

X_df = pd.DataFrame(X)
Y_df = pd.DataFrame(Y)


def get_integer_columns(X, tol=1e-5):
    """
    判断 X 中哪些列为整数列。

    参数:
        X (np.ndarray): 形状为 (n_samples, n_features) 的数组
        tol (float): 容差，若每个数值与其四舍五入后的差小于 tol，则认为该数值为整数

    返回:
        int_cols (list): 整数列的列索引列表
    """
    int_cols = []
    for i in range(X.shape[1]):
        col = X[:, i]
        # 判断该列所有数值是否接近整数
        if np.all(np.abs(col - np.round(col)) < tol):
            int_cols.append(i)
    return int_cols

# integer_indices = get_integer_columns(X)

# print("整数列的索引为:", integer_indices)

# 如果需要查看这些列的内容，可以转换为 DataFrame 查看前几行：

# X_int = pd.DataFrame(X[:, integer_indices])
X_df.head(5)

In [ ]:
X_int.head(5)

In [ ]:
X_df.head(5)

In [ ]:
import pandas as pd
import numpy as np

# 路径
X_path = "/data/dengqiyan/EVA/data/IMDB/IMDBLargeC5-train-X.npy"
new_path = "/data/dengqiyan/EVA/data/IMDB/new/IMDBLargeC5-train-X.npy"

# 加载数据
X = np.load(X_path)

# 转换为 DataFrame
X_df = pd.DataFrame(X)

# 提取第0列和第7列
cols_to_move = X_df.iloc[:, [0, 7]]

# 删除第0列和第7列
X_df = X_df.drop([0, 7], axis=1)

# 将第0列和第7列添加到最后
X_df = pd.concat([X_df, cols_to_move], axis=1)

# 转回 NumPy 数组
X_new = X_df.values

# 保存新的 .npy 文件
np.save(new_path, X_new)

print(f"新数据已保存至: {new_path}")